# NB2b — Algorithmic Fact Extraction

In this notebook I extract the "hard facts" from each human article using purely
algorithmic, model-free methods. This is **Track B** of my dual-track summarization
design (Track A is the ALLaM-based structured extraction in NB2b/NB2a).

**Input:** `ha_corpus.parquet` — my 3,500 cleaned human articles from NB1b.
**Output:** `extract_algo.parquet` — feeds into NB2c, where I merge both tracks into fact cards.

## Why I chose an algorithmic pipeline over an LLM here

I deliberately keep this track LLM-free for four reasons:

| Property | Why it matters for my design |
|---|---|
| **Fully neutral** | It injects no model's stylistic fingerprint (unlike an LLM summarizer) |
| **Reproducible** | Same input always yields the same output — no sampling randomness |
| **No hallucination** | It only quotes elements that actually exist in the source text |
| **Fast / free** | Minutes, not hours; no API cost |

## What I extract (five components)

1. **Entities** (Arabic NER — CAMeL): persons, locations, organizations → always required.
2. **Numbers & dates** (regex) → mirror rule.
3. **News agencies** (dictionary match) → mirror rule.
4. **Key sentences** (TextRank) → basis for my fact points.
5. **Quote events** (regex): the *fact* of a statement, not its verbatim text → mirror rule.

**My mirror rule:** I only extract what is present in the source; any absent field stays
empty and is never later forced onto the generator. This keeps the AI class from
diverging from the human class on cheap surface signals.

In [1]:
import pandas as pd
import numpy as np
import re
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

# Kaggle paths — I set IN_PATH to my uploaded ha_corpus dataset
IN_PATH  = '/kaggle/input/notebooks/bahaaqassem/nb0b-select-corpus/ha_corpus.parquet'
OUT_DIR  = '/kaggle/working'

df = pd.read_parquet(IN_PATH)
print('shape:', df.shape)
print('columns:', df.columns.tolist())
print(df[['id','length_words']].head())

shape: (3500, 5)
columns: ['id', 'text', 'length_words', 'source_domain', 'url']
         id  length_words
0  HA_00000           458
1  HA_00001           465
2  HA_00002           444
3  HA_00003           458
4  HA_00004           423


## Component 1: Sentence splitting + fact-point count by length

I protect common abbreviations (`د.`, `أ.`) from being wrongly split.
I scale the number of fact points to article length:
`clip(round(words / 400), 3, 7)` — minimum 3, maximum 7 (so short articles still
get a clear topic, and very long ones don't drift into full rewriting).

In [2]:
SENT_SPLIT = re.compile(r'(?<=[\.\!\؟\?])(?<!\bد\.)(?<!\bأ\.)(?<!\bص\.)(?<!\bم\.)\s+')

def split_sentences(text, min_words=4):
    return [s.strip() for s in SENT_SPLIT.split(str(text)) if len(s.split()) >= min_words]

def n_fact_points(wc):
    return int(np.clip(round(wc / 400), 3, 7))

## Component 2: Numbers & dates (mirror rule)

In [3]:
MONTHS = ('كانون|تشرين|شباط|آذار|نيسان|أيار|حزيران|تموز|آب|أيلول|'
          'يناير|فبراير|مارس|أبريل|مايو|يونيو|يوليو|أغسطس|سبتمبر|أكتوبر|نوفمبر|ديسمبر')

def extract_numbers_dates(text, cap=20):
    items = []
    items += re.findall(r'(?:\d{1,2}\s*)?(?:' + MONTHS + r')(?:\s*\d{4})?', text)  # dates
    items += re.findall(r'\b\d{1,4}(?:[.,]\d+)?\b', text)                          # numbers
    seen, out = set(), []
    for it in items:
        it = it.strip()
        if it and it not in seen:
            seen.add(it); out.append(it)
    return out[:cap]

## Component 3: News agencies (mirror rule)

A dictionary of common Arabic news agencies. I drop overlaps
(e.g. "الجزيرة نت" supersedes "الجزيرة").

In [4]:
AGENCIES = ['الجزيرة نت','الجزيرة','رويترز','فرانس برس','الأناضول','أ ف ب','قنا',
            'وكالة الأنباء','أسوشيتد برس','الوكالة الفلسطينية','وفا','سبوتنيك',
            'العربية','سكاي نيوز','بي بي سي','الشرق الأوسط','واس','بترا']

def extract_agencies(text):
    found = [a for a in AGENCIES if a in text]
    if 'الجزيرة نت' in found and 'الجزيرة' in found:   # longer form wins
        found.remove('الجزيرة')
    return sorted(set(found))

## Component 4: Key sentences (TextRank)

I build a similarity graph over sentences (TF-IDF + cosine) and run PageRank to pick
the top `k` sentences. I return them in their original order of appearance so the
downstream fact card reads in a natural sequence.

In [5]:
def textrank_key_sentences(text, k):
    sents = split_sentences(text)
    if len(sents) <= k:
        return sents
    try:
        tfidf = TfidfVectorizer().fit_transform(sents)
        sim = cosine_similarity(tfidf)
        np.fill_diagonal(sim, 0)
        g = nx.from_numpy_array(sim)
        scores = nx.pagerank(g, max_iter=100)
        ranked = sorted(range(len(sents)), key=lambda i: scores[i], reverse=True)[:k]
        return [sents[i] for i in sorted(ranked)]   # keep original order
    except Exception:
        return sents[:k]

## Component 5: Quote events (mirror rule)

I capture the *event* of a statement (reporting verb + a short span), not the long
verbatim quote — so the generator re-phrases it as its own quote instead of copying it.
This prevents verbatim quotes from becoming a cheap human/AI signal.

In [6]:
QUOTE_VERBS = r'قال|أضاف|أكد|صرح|أوضح|أشار|ذكر|أعلن|كشف|نفى|شدد|لفت|بيّن|اعتبر|أفاد|روى'

def extract_quote_events(text, cap=5):
    events = []
    for m in re.finditer(r'(?:و)?(?:' + QUOTE_VERBS + r')\s+([^\.،؟!]{5,60})', text):
        events.append(m.group(0).strip()[:70])
    return list(dict.fromkeys(events))[:cap]

## Component 6: Arabic NER (CAMeL) — robust version

I extract the core entities (persons / locations / organizations) with
`CAMeL-Lab/bert-base-arabic-camelbert-msa-ner` — the **same** model I'll later use for the
Entity-Density feature, so the two stay consistent. This step needs GPU + Internet.

**Fixes across my runs (this is the clean version):**
- **Chunk + union:** truncating loses ~90% of long-article entities, so I chunk each article
  and union the entity lists (safe for NER — entities are recognized locally).
- **`aggregation_strategy='first'`:** the documented choice for morphologically rich Arabic;
  it merges a word's sub-tokens properly (fixes fragments like "فة الغربية" → "الضفة").
  Switchable to `'max'` in one line if I want to compare.
- **Word-aligned chunk boundaries:** I pack whole words up to the token limit so a word is
  never split across two chunks (that split produced fragments like "ئب عريقات").
- **Stronger fragment filter:** I drop `[UNK]`, leftover `##`, tails starting with ة/ى/ء,
  and stray short pieces — while keeping a whitelist of valid short entities (غزة, رفح ...).
- **Tag normalization + agency removal + form dedup** as before.

In [7]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

# Aggregation strategy: 'first' is the documented choice for morphologically rich
# languages like Arabic (it takes the class of the word's first sub-token, which best
# represents the word). 'max' is a valid alternative — I can switch this one word and re-run.
AGG_STRATEGY = 'first'

ner = pipeline('ner',
               model='CAMeL-Lab/bert-base-arabic-camelbert-msa-ner',
               aggregation_strategy=AGG_STRATEGY,
               device=device,
               batch_size=16)

# CAMeL NER is a BERT model with a hard 512-token limit. Fixes:
#   - I CHUNK long articles (truncating loses ~90% of long-article entities) and UNION results.
#   - I align chunk boundaries on WORD boundaries (not raw tokens), so a word is never split
#     across two chunks (that split was one source of fragmented entities like "ئب عريقات").
NER_CHUNK_TOKENS = 500
NER_OVERLAP_WORDS = 20     # word-level overlap between chunks

def chunk_by_words(text, max_tokens=NER_CHUNK_TOKENS, overlap_words=NER_OVERLAP_WORDS):
    # Split into words, then greedily pack whole words until we approach the token limit.
    words = str(text).split()
    if not words:
        return ['']
    chunks, cur, cur_tok = [], [], 0
    i = 0
    while i < len(words):
        w = words[i]
        wt = len(ner.tokenizer.encode(w, add_special_tokens=False)) or 1
        if cur and cur_tok + wt > max_tokens:
            chunks.append(' '.join(cur))
            # start next chunk with a word-level overlap for boundary safety
            cur = cur[-overlap_words:] if overlap_words else []
            cur_tok = sum(len(ner.tokenizer.encode(x, add_special_tokens=False)) or 1 for x in cur)
        cur.append(w); cur_tok += wt; i += 1
    if cur:
        chunks.append(' '.join(cur))
    return chunks

# --- entity post-processing helpers ---
PREFIXES = ['لـ','بـ','كـ','فـ']   # only clear space-separated prefixes; I don't risk cutting ال

# Short but valid entities I don't want the fragment filter to drop
VALID_SHORT = {'رفح','غزة','فتح','حماس','دمشق','لندن','مصر','عمان','صنعاء','بغداد','تونس',
               'الأقصى','القدس','حلب','حمص','درعا','إدلب','العراق','لبنان','سوريا','ليبيا',
               'اليمن','قطر','حماة','نابلس','جنين','الخليل','بيروت','طهران','أنقرة','برلين',
               'باريس','موسكو','واشنطن','الرباط','تل أبيب'}

def normalize_group(g):
    # Map any tag form (PERS / B-PERS / per / GPE ...) to my 3 categories
    g = g.upper().replace('B-','').replace('I-','').strip()
    if g.startswith('PER'): return 'persons'
    if g.startswith('LOC') or g.startswith('GPE'): return 'locations'
    if g.startswith('ORG'): return 'organizations'
    return None

def clean_entity(e):
    e = e.replace('##','').strip()          # safety: drop any leftover WordPiece marker
    for p in PREFIXES:
        if e.startswith(p) and len(e) > len(p)+2:
            e = e[len(p):].strip(); break
    return e

def is_valid_entity(e):
    if '[UNK]' in e or '##' in e:            # unknown token or leftover fragment marker
        return False
    if len(e) < 3: return False
    if e in ('ال','الـ'): return False
    if e[0] in 'ةىءئؤ':                       # a word can't start with these → fragment tail
        return False
    parts = e.split()
    # first word of a multi-word entity is a short fragment (e.g. "فة الغربية") → drop
    if len(parts) > 1 and len(parts[0]) <= 2:
        return False
    # single short token that isn't a known short entity → likely a fragment
    if len(parts) == 1 and len(e) <= 3 and e not in VALID_SHORT:
        return False
    return True

def dedupe_forms(items):
    # Merge partial forms: drop a token that is part of a longer kept name
    items = sorted(set(items), key=len, reverse=True)
    kept = []
    for e in items:
        if any(e == k or e in k.split() for k in kept):
            continue
        kept.append(e)
    return kept

def parse_ner(raw_ents):
    out = {'persons': [], 'locations': [], 'organizations': []}
    for e in raw_ents:
        key = normalize_group(e.get('entity_group',''))
        if not key:
            continue
        w = clean_entity(e.get('word',''))
        if not is_valid_entity(w):
            continue
        if any(a in w or w in a for a in AGENCIES):   # agencies live in source_agencies
            continue
        if w not in out[key]:
            out[key].append(w)
    return {k: dedupe_forms(v) for k, v in out.items()}

config.json:   0%|          | 0.00/980 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertForTokenClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa-ner
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

### Diagnostic — confirm the tag format before the full run

Before processing all 3,500, I run NER on one article that came back empty last time and
**print the raw tags**. This tells me for certain what `entity_group` values the model emits,
so I know my normalization covers them. (Takes seconds.)

In [8]:
# One-off diagnostic on a previously-empty article.
# I run NER on its chunks and print the raw tags to confirm the format and that entities appear.
_chunks = chunk_by_words(df[df['id'] == 'HA_00001']['text'].values[0])
_raw = []
for _c in _chunks:
    _raw.extend(ner(_c))
print('num chunks:', len(_chunks))
print('raw entity_group values seen:', sorted(set(e['entity_group'] for e in _raw)))
print('raw count:', len(_raw))
print('sample raw:', [(e['entity_group'], e['word']) for e in _raw[:8]])
print('after parse_ner:', parse_ner(_raw))

num chunks: 2
raw entity_group values seen: ['LOC', 'ORG', 'PERS']
raw count: 56
sample raw: [('LOC', 'حلب'), ('LOC', 'موسكو'), ('LOC', 'ودمشق'), ('LOC', 'حلب'), ('PERS', 'وائل الحلقي'), ('LOC', 'حلب'), ('LOC', 'حلب'), ('ORG', 'إنترفاكس')]
after parse_ner: {'persons': ['وائل الحلقي', 'أديب عليوي', 'محمد خطاب'], 'locations': ['بسوريا', 'وروسيا', 'وموسكو', 'موسكو', 'روسيا', 'إيران', 'ودمشق', 'سوريا', 'دمشق', 'إدلب', 'جنيف', 'حلب'], 'organizations': ['وجبهة النصرة', 'جبهة النصرة', 'للجزيرة نت', 'إنترفاكس']}


### Entity extraction over all articles (chunk + union)

For each article I chunk it into 500-token windows, run NER on all chunks, and union the
entity lists. Short articles are a single chunk; long articles keep all their entities
(including tail entities the old truncation dropped). I flatten all chunks across a batch
into one NER call for speed, then regroup per article.

In [9]:
def extract_entities_for_texts(texts):
    # Build the chunk list for every text, remembering how many chunks each produced
    all_chunks, owner = [], []
    for idx, t in enumerate(texts):
        chs = chunk_by_words(t)
        all_chunks.extend(chs)
        owner.extend([idx] * len(chs))

    # One batched NER call over every chunk
    try:
        raw = ner(all_chunks)
        if raw and isinstance(raw[0], dict):     # single-chunk safeguard
            raw = [raw]
    except Exception as ex:
        print('NER batch error:', ex, flush=True)
        raw = [[] for _ in all_chunks]

    # Regroup chunk results back to their owning article, then parse+union
    per_article = [[] for _ in texts]
    for ch_i, ents in enumerate(raw):
        per_article[owner[ch_i]].extend(ents)
    return [parse_ner(ents) for ents in per_article]

## Main loop — fault-tolerant, incremental save, clear log

I run this via **Save Version** in the background because of frequent power cuts on my
side. So I built the loop to survive:

- **Every article is wrapped in try/except** — if one article fails (a transient NER
  error, a bad character), I log it and keep going. A single failure never kills the run.
- **Incremental save every 100 articles** to `/kaggle/working`, so the output survives
  even if a later cell crashes.
- **Clear progress prints** (with `flush=True`) so I can follow the live log and see
  exactly where the run is and whether anything failed.

**Resuming across versions:** if I need to restart, I attach the previous version's
output as an input dataset and point `RESUME_PATH` at it; the run continues where it stopped.

In [10]:
import os, time

CKPT        = f'{OUT_DIR}/extract_algo.parquet'   # incremental save == final output
SAVE_EVERY  = 100        # save every N articles
NER_BATCH   = 16         # articles per NER batch
RESUME_PATH = None       # e.g. '/kaggle/input/aigt-extract-algo-partial/extract_algo.parquet'

# Resume from a previous version's output or from an in-session checkpoint
done_ids, records = set(), []
for p in [RESUME_PATH, CKPT]:
    if p and os.path.exists(p):
        prev = pd.read_parquet(p)
        records = prev.to_dict('records')
        done_ids = set(prev['id'])
        print(f'Resuming from {p}: {len(done_ids)} articles already done')
        break

todo = df[~df['id'].isin(done_ids)].reset_index(drop=True)
print(f'total: {len(df)} | done: {len(done_ids)} | remaining: {len(todo)}', flush=True)

t0 = time.time()
n_ok, n_err = 0, 0

# Process in batches so NER runs on a list of texts (fixes the empty-return bug)
for start in range(0, len(todo), NER_BATCH):
    batch = todo.iloc[start:start + NER_BATCH]
    texts = batch['text'].tolist()

    # Chunked+unioned NER for the whole group at once
    try:
        ent_list = extract_entities_for_texts(texts)
    except Exception as ex:
        print(f'WARN NER batch failed at {start}: {ex}', flush=True)
        ent_list = [{'persons': [], 'locations': [], 'organizations': []} for _ in texts]

    # Per-article: the fast algorithmic parts + attach the batched entities
    for j, (_, row) in enumerate(batch.iterrows()):
        try:
            text = str(row['text']); wc = int(row['length_words'])
            k = n_fact_points(wc)
            rec = {
                'id': row['id'],
                'pair_id': row['id'],                 # becomes source_pair_id of the AI counterpart
                'entities':        json.dumps(ent_list[j], ensure_ascii=False),
                'numbers_dates':   json.dumps(extract_numbers_dates(text), ensure_ascii=False),
                'source_agencies': json.dumps(extract_agencies(text),      ensure_ascii=False),
                'key_sentences':   json.dumps(textrank_key_sentences(text, k), ensure_ascii=False),
                'quote_events':    json.dumps(extract_quote_events(text),  ensure_ascii=False),
                'n_fact_points':   k,
                'target_words':    wc,
                'error':           '',
            }
            n_ok += 1
        except Exception as e:
            rec = {'id': row['id'], 'pair_id': row['id'], 'entities':'{}',
                   'numbers_dates':'[]','source_agencies':'[]','key_sentences':'[]',
                   'quote_events':'[]','n_fact_points':0,'target_words':int(row['length_words']),
                   'error': f'{type(e).__name__}: {str(e)[:150]}'}
            n_err += 1
            print(f'WARN failed {row["id"]}: {rec["error"]}', flush=True)
        records.append(rec)

    # Incremental save + progress log
    processed = len(records)
    if (processed % SAVE_EVERY < NER_BATCH) or (start + NER_BATCH >= len(todo)):
        pd.DataFrame(records).to_parquet(CKPT, index=False)
        rate = (processed - len(done_ids)) / max(time.time() - t0, 1)
        print(f'  saved {processed}/{len(df)} | ok={n_ok} err={n_err} | ~{rate:.1f} art/s', flush=True)

print(f'\nDONE algorithmic extraction | ok={n_ok} err={n_err}', flush=True)

total: 3500 | done: 0 | remaining: 3500
  saved 112/3500 | ok=112 err=0 | ~9.7 art/s


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  saved 208/3500 | ok=208 err=0 | ~10.0 art/s
  saved 304/3500 | ok=304 err=0 | ~10.0 art/s
  saved 400/3500 | ok=400 err=0 | ~10.0 art/s
  saved 512/3500 | ok=512 err=0 | ~10.0 art/s
  saved 608/3500 | ok=608 err=0 | ~10.0 art/s
  saved 704/3500 | ok=704 err=0 | ~9.9 art/s
  saved 800/3500 | ok=800 err=0 | ~9.8 art/s
  saved 912/3500 | ok=912 err=0 | ~9.7 art/s
  saved 1008/3500 | ok=1008 err=0 | ~9.6 art/s
  saved 1104/3500 | ok=1104 err=0 | ~9.6 art/s
  saved 1200/3500 | ok=1200 err=0 | ~9.5 art/s
  saved 1312/3500 | ok=1312 err=0 | ~9.4 art/s
  saved 1408/3500 | ok=1408 err=0 | ~9.3 art/s
  saved 1504/3500 | ok=1504 err=0 | ~9.2 art/s
  saved 1600/3500 | ok=1600 err=0 | ~9.1 art/s
  saved 1712/3500 | ok=1712 err=0 | ~9.0 art/s
  saved 1808/3500 | ok=1808 err=0 | ~9.0 art/s
  saved 1904/3500 | ok=1904 err=0 | ~8.9 art/s
  saved 2000/3500 | ok=2000 err=0 | ~8.9 art/s
  saved 2112/3500 | ok=2112 err=0 | ~8.8 art/s
  saved 2208/3500 | ok=2208 err=0 | ~8.6 art/s
  saved 2304/3500 | ok=2

In [11]:
# Reload the incrementally-saved file for a final check
final = pd.read_parquet(f'{OUT_DIR}/extract_algo.parquet')
print('shape:', final.shape)

# Error report (if any)
n_err = (final['error'] != '').sum()
print(f'failed articles: {n_err}')
if n_err:
    print(final[final['error'] != ''][['id','error']].head(20).to_string())

# Spot-check a few successful rows
import json as _j
ok = final[final['error'] == '']
for i in range(min(3, len(ok))):
    r = ok.iloc[i]
    print(f"\n=== {r['id']} (points={r['n_fact_points']}) ===")
    print('  entities:', _j.loads(r['entities']))
    print('  agencies:', _j.loads(r['source_agencies']))
    print('  numbers/dates:', _j.loads(r['numbers_dates'])[:5])
    print('  key sentences:', len(_j.loads(r['key_sentences'])))

# Coverage stats
print('\n--- stats ---')
def _has_entities(x):
    d = _j.loads(x); return sum(len(v) for v in d.values()) > 0
n_with_ent = ok['entities'].apply(_has_entities).sum()
print('with entities:', n_with_ent, '/', len(ok),
      f'({100*n_with_ent/max(len(ok),1):.0f}%)   <-- must be high now')
print('with agencies:', (ok['source_agencies'] != '[]').sum(), '/', len(ok))
print('with quotes:', (ok['quote_events'] != '[]').sum(), '/', len(ok))
print('with numbers/dates:', (ok['numbers_dates'] != '[]').sum(), '/', len(ok))

# avg clean entities per article
import numpy as _np
avg_ent = ok['entities'].apply(lambda x: sum(len(v) for v in _j.loads(x).values())).mean()
print(f'avg entities/article: {avg_ent:.1f}')

shape: (3500, 10)
failed articles: 0

=== HA_00000 (points=3) ===
  entities: {'persons': ['رجب طيب أردوغان', 'صياء عبد الرضا', 'جورج دبليو بوش', 'دافيد بارنياع', 'أنتوني بلينكن', 'دافيد برنياع', 'نفتالي بينيت', 'موشيه يعلون', 'بيني غانتس'], 'locations': ['الولايات المتحدة', 'وفنلندا', 'إسرائيل', 'أمريكا', 'واشنطن', 'لإيران', 'السويد', 'إيران', 'ايران', 'تركيا', 'طهران', 'فيينا'], 'organizations': ['حلف شمال الأطلسي', 'يديعوت أحرونوت', 'الغارديان', 'الموساد']}
  agencies: ['الشرق الأوسط', 'قنا', 'واس']
  numbers/dates: ['مارس', '60', '3', '1980']
  key sentences: 3

=== HA_00001 (points=3) ===
  entities: {'persons': ['وائل الحلقي', 'أديب عليوي', 'محمد خطاب'], 'locations': ['بسوريا', 'وروسيا', 'وموسكو', 'موسكو', 'روسيا', 'إيران', 'ودمشق', 'سوريا', 'دمشق', 'إدلب', 'جنيف', 'حلب'], 'organizations': ['وجبهة النصرة', 'جبهة النصرة', 'للجزيرة نت', 'إنترفاكس']}
  agencies: ['واس']
  numbers/dates: []
  key sentences: 3

=== HA_00002 (points=3) ===
  entities: {'persons': ['بنيامين نتنياهو'], '

## Notes & known limits

- **NER on the first 2,500 chars:** in news writing the core entities appear early
  (inverted pyramid), so I cap the span to stay within the model limit and speed things up.
- **Agencies via a fixed list:** a rare agency might be missed; the list covers the common
  ones and is easy to extend.
- **Quote events are approximate:** I capture the "reporting verb + span" pattern; the goal
  is to describe the statement, not reproduce it exactly.
- **Next step (NB2c):** I merge this output with the ALLaM track and build the fact cards,
  cross-checking the entities from both tracks.

I upload `extract_algo.parquet` as a Kaggle dataset (e.g. `aigt-extract-algo`) so NB2c can use it.